# Final Notebook — Fraud Risk → Business Action
### IEEE-CIS primary · multi-model · 5-seed mean±std · PCA/UMAP · capacity-aware cost policy

**Revision log (this version fixes the methodological issues flagged in review):**

| # | Issue | Fix |
|---|---|---|
| 1 | Cost/action thresholds were optimized directly on TEST | Thresholds are now optimized on **VAL**, frozen, then evaluated once on **TEST** |
| 2 | "Limited review capacity" was never enforced | `optimize_thresholds_capacity()` adds a hard `review_rate ≤ budget` constraint (1% / 5% / 10%) |
| 3 | No baseline-vs-behavioral ablation existed | **EXP-0** (IEEE-CIS native features) vs **EXP-1** (+ leakage-safe behavioral features), same model/seed protocol, reported as Δ |
| 4 | `beh_prev_fraud_rate` / `beh_prev_fraud_count` assume the previous transaction's fraud label is known instantly | Excluded from the **main** deployment-realistic feature set by default; kept only as an optional **label-aware sensitivity** experiment |
| 5 | `beh_amount_ratio` median and `beh_time_since_prev` max were computed over the *whole* dataframe before the split | Both are now left as `NaN` and imputed by the `Preprocessor`, which is **fit on TRAIN only** |
| 6 | High-missingness column dropping was computed over the whole dataframe before the split | Moved to *after* the temporal split; missingness is computed on **TRAIN only** and the same columns are dropped from VAL/TEST |
| 7 | Ensemble threshold was `mean(per-model VAL thresholds)`, not calibrated on ensemble VAL scores | VAL probabilities are now stored per (model, seed); the ensemble threshold is tuned on the **ensemble's own VAL scores**, then frozen for TEST |
| 8 | SMOTE was mentioned in the study description but absent from the notebook | Added as an explicit, optional secondary sensitivity experiment (Section 11) |
| 9 | PaySim was mentioned but not present | Left out; documented as a future cross-dataset transfer study, not claimed here |

**Includes**
- Composite entity key + leakage-safe behavioral features (deployment-realistic by default)
- Temporal train/val/test split (no random leakage)
- LightGBM + XGBoost + RandomForest, **5 seeds**, mean ± std
- **VAL-tuned F1 threshold** applied to TEST
- Behavioral ablation (EXP-0 vs EXP-1)
- Soft ensemble calibrated on VAL, evaluated once on TEST
- PCA + UMAP with fraud emphasis + covariance ellipse (illustrative only)
- **Capacity- and cost-aware** APPROVE / REVIEW / BLOCK policy, tuned on VAL, evaluated on TEST
- FC@1% / FC@5% / FC@10%
- SHAP (global + representative high/med/low-risk transactions)
- Optional SMOTE sensitivity experiment

**Metrics note:** PR-AUC / ROC-AUC are ranking metrics. Precision/Recall/F1/MCC use a threshold chosen on VAL (max F1), then applied to TEST. Cost-policy thresholds are chosen on VAL under a review-capacity constraint, then frozen and applied to TEST.


## 0. Setup

In [ ]:
import os, gc, json, time, warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, precision_recall_curve,
)

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("Install lightgbm")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost optional")

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("Optional: !pip install umap-learn")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("Optional: !pip install imbalanced-learn  (needed for Section 11 SMOTE sensitivity)")

RANDOM_STATE = 42
SEEDS = [42, 7, 123, 2024, 99]
np.random.seed(RANDOM_STATE)

def rss_mb():
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024.0
    except Exception:
        pass
    return -1.0

print(f"LGB={HAS_LGB} XGB={HAS_XGB} UMAP={HAS_UMAP} SHAP={HAS_SHAP} IMBLEARN={HAS_IMBLEARN} RSS={rss_mb():.0f}")


## 1. CONFIG

In [ ]:
CONFIG = {
    "ieee_transaction_path": "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv",
    "ieee_identity_path": "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv",

    # 300k default for fast iteration; set to None for the final full-data paper run.
    # NOTE: because the split is temporal, a head-of-file sample under-represents later
    # transactions. Document this explicitly if the final run keeps a sample.
    "SAMPLE_N_IEEE": 300_000,

    "train_frac": 0.60,
    "val_frac": 0.20,
    "test_frac": 0.20,

    # High-missingness column dropping is now fit on TRAIN ONLY (see Section 3).
    "drop_high_missing": True,
    "high_missing_threshold": 0.92,
    "velocity_windows": [3600 * h for h in [1, 6, 12, 24, 72, 168]],

    "output_dir": "/kaggle/working/outputs_final",
    "feat_parquet": "/kaggle/working/outputs_final/ieee_feat.parquet",

    # ---------------- methodological fixes ----------------
    # Main pipeline excludes previous-transaction fraud labels (unrealistic to assume
    # instant label confirmation at scoring time). Only used in the optional
    # label-aware sensitivity experiment in Section 5c.
    "USE_HISTORICAL_FRAUD_LABELS_MAIN": False,
    "RUN_FRAUD_LABEL_SENSITIVITY": True,

    # Operational cost/action policy: optimized on VAL under a review-capacity
    # constraint, then frozen and evaluated once on TEST.
    "REVIEW_CAPACITY_BUDGETS": [0.01, 0.05, 0.10],
    "COST_SCENARIOS": {"S1_10x": 10.0, "S2_25x": 25.0, "S3_50x": 50.0},
    "C_FP": 1.0,
    "C_REVIEW": 0.2,

    # Optional secondary experiments
    "RUN_SMOTE_SENSITIVITY": True,
    "RUN_BASELINE_ABLATION": True,  # EXP-0 vs EXP-1
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if "path" not in k}, indent=2))


## 2. Load IEEE-CIS + composite entity

*(High-missingness column dropping has been moved to Section 3, fit on TRAIN only — see Issue 6.)*

In [ ]:
def downcast_df(df):
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def load_ieee(config):
    t0 = time.time()
    nrows = config["SAMPLE_N_IEEE"]
    print(f"Loading transaction nrows={nrows}...", flush=True)
    df = pd.read_csv(config["ieee_transaction_path"], nrows=nrows)
    df = downcast_df(df)

    ident_path = Path(config["ieee_identity_path"])
    if ident_path.exists():
        ids = set(df["TransactionID"].values)
        chunks = []
        for ch in pd.read_csv(ident_path, chunksize=250_000):
            ch = ch[ch["TransactionID"].isin(ids)]
            if len(ch):
                chunks.append(ch)
        if chunks:
            ident = downcast_df(pd.concat(chunks, ignore_index=True))
            df = df.merge(ident, on="TransactionID", how="left")
            del ident, chunks
            gc.collect()

    # NOTE: high-missingness dropping intentionally NOT done here anymore.
    # It is done in Section 3, fit on TRAIN only, to avoid using VAL/TEST
    # missingness statistics to decide which columns exist in the model.

    parts = [df["card1"].astype(str).fillna("NA")]
    if "addr1" in df.columns:
        parts.append(df["addr1"].astype(str).fillna("NA"))
    if "P_emaildomain" in df.columns:
        parts.append(df["P_emaildomain"].astype(str).fillna("NA"))
    df["entity_key"] = parts[0]
    for p in parts[1:]:
        df["entity_key"] = df["entity_key"] + "|" + p

    df = df.sort_values("TransactionDT").reset_index(drop=True)
    df = downcast_df(df)
    print(f"Loaded {df.shape} fraud={df['isFraud'].mean()*100:.3f}% in {time.time()-t0:.1f}s RSS={rss_mb():.0f}")
    return df

df = load_ieee(CONFIG)


## 3. Leakage-safe behavioral features

`beh_prev_fraud_rate` / `beh_prev_fraud_count` are still **computed** here (cheap, and useful for the
optional sensitivity experiment) but are **excluded from the main feature set** in Section 4, because they
assume the previous transaction's fraud label is already confirmed at scoring time — an unrealistic
assumption for most fraud-ops pipelines (Issue 4).

`beh_amount_ratio` and `beh_time_since_prev` no longer get their missing values filled with a
whole-dataframe median/max (Issue 5) — they are left as `NaN` and imputed later by the `Preprocessor`,
which is fit on **TRAIN only**.

In [ ]:
def add_behavioral_features(df, entity_col="entity_key", amount_col="TransactionAmt",
                            time_col="TransactionDT", windows=None, eps=1e-6):
    windows = windows or CONFIG["velocity_windows"]
    t0 = time.time()
    df = df.sort_values(time_col).reset_index(drop=True)
    g = df.groupby(entity_col, sort=False)

    df["beh_prev_tx_count"] = g.cumcount().astype(np.int32)
    shifted = g[amount_col].shift(1)
    past_cnt = df["beh_prev_tx_count"].astype(np.float32)
    past_sum = shifted.fillna(0).groupby(df[entity_col], sort=False).cumsum()
    df["beh_prev_amount_mean"] = np.where(past_cnt > 0, past_sum / past_cnt, np.nan).astype(np.float32)

    past_sum_sq = (shifted ** 2).fillna(0).groupby(df[entity_col], sort=False).cumsum()
    mean_sq = np.where(past_cnt > 0, past_sum_sq / past_cnt, np.nan)
    var = np.maximum(mean_sq - df["beh_prev_amount_mean"].astype(np.float64) ** 2, 0)
    df["beh_prev_amount_std"] = np.sqrt(var).astype(np.float32)  # NaN where no history yet (fine)

    df["beh_amount_zscore"] = ((df[amount_col] - df["beh_prev_amount_mean"]) / (df["beh_prev_amount_std"] + eps)).astype(np.float32)
    df["beh_amount_ratio"] = (df[amount_col] / (df["beh_prev_amount_mean"] + eps)).astype(np.float32)
    first = df["beh_prev_tx_count"] == 0
    df.loc[first, ["beh_amount_zscore", "beh_amount_ratio"]] = 0.0
    df["beh_amount_ratio"] = df["beh_amount_ratio"].replace([np.inf, -np.inf], np.nan)
    # FIX (Issue 5): do NOT fill with a whole-dataframe median here — that leaks VAL/TEST
    # distribution into the imputation value. Leave as NaN; Preprocessor (TRAIN-only fit)
    # imputes it downstream.

    df["beh_time_since_prev"] = (df[time_col] - g[time_col].shift(1)).astype(np.float32)
    # FIX (Issue 5): do NOT fill with the whole-dataframe max here, same reasoning as above.
    # Leave as NaN for the Preprocessor to impute from TRAIN.

    df["beh_log_amount"] = np.log1p(df[amount_col].astype(np.float32))
    tod = (df[time_col] % 86400).astype(np.float32)
    df["beh_hour_sin"] = np.sin(2 * np.pi * tod / 86400).astype(np.float32)
    df["beh_hour_cos"] = np.cos(2 * np.pi * tod / 86400).astype(np.float32)
    df["beh_dow_sin"] = np.sin(2 * np.pi * (df[time_col] % (86400 * 7)) / (86400 * 7)).astype(np.float32)
    df["beh_dow_cos"] = np.cos(2 * np.pi * (df[time_col] % (86400 * 7)) / (86400 * 7)).astype(np.float32)

    if "isFraud" in df.columns:
        # Computed for completeness / the optional label-aware sensitivity experiment.
        # NOT included in the main deployment-realistic feature set (see Section 4).
        shifted_y = g["isFraud"].shift(1)
        past_fraud_sum = shifted_y.fillna(0).groupby(df[entity_col], sort=False).cumsum()
        df["beh_prev_fraud_rate"] = np.where(past_cnt > 0, past_fraud_sum / past_cnt, 0).astype(np.float32)
        df["beh_prev_fraud_count"] = past_fraud_sum.astype(np.float32)

    print("  velocity...", flush=True)
    entities = df[entity_col].fillna("__NA__").values
    times = df[time_col].values
    order = np.argsort(entities, kind="stable")
    ent_vals, t_vals = entities[order], times[order]
    for w in windows:
        counts = np.zeros(len(df), dtype=np.int32)
        n, i = len(df), 0
        while i < n:
            j = i
            while j < n and ent_vals[j] == ent_vals[i]:
                j += 1
            sub = t_vals[i:j]
            lo = 0
            for k in range(len(sub)):
                cutoff = sub[k] - w
                while lo < k and sub[lo] < cutoff:
                    lo += 1
                counts[i + k] = k - lo
            i = j
        out = np.empty(len(df), dtype=np.int32)
        out[order] = counts
        df[f"beh_velocity_{w}s"] = out

    print(f"  done {time.time()-t0:.1f}s shape={df.shape} RSS={rss_mb():.0f}")
    return df

df = add_behavioral_features(df)
df.to_parquet(CONFIG["feat_parquet"], index=False)
print("Saved", CONFIG["feat_parquet"])
del df
gc.collect()


## 4. Temporal split + preprocess

High-missingness columns are now dropped **after** the split, fitting missingness on **TRAIN only**
(Issue 6), then the same columns are dropped from VAL/TEST. Two feature lists are built:

- **EXP-0 baseline**: native IEEE-CIS features only (no `beh_*`)
- **EXP-1 behavioral (main)**: baseline + leakage-safe behavioral features, **excluding** `beh_prev_fraud_rate` / `beh_prev_fraud_count`

A third, `beh_fraud_history`, feature list (baseline + behavioral + fraud-history features) is built for the
optional label-aware sensitivity experiment in Section 5c.

In [ ]:
df = pd.read_parquet(CONFIG["feat_parquet"])
print("Loaded", df.shape, f"RSS={rss_mb():.0f}")

def temporal_split(df, time_col="TransactionDT"):
    df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df)
    a = int(n * CONFIG["train_frac"])
    b = int(n * (CONFIG["train_frac"] + CONFIG["val_frac"]))
    tr, va, te = df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()
    for name, d in [("train", tr), ("val", va), ("test", te)]:
        print(f"  {name}: {len(d):,} fraud={d['isFraud'].mean()*100:.3f}%")
    return tr, va, te

train_df, val_df, test_df = temporal_split(df)
del df
gc.collect()

# ---- FIX (Issue 6): drop high-missingness columns, fit on TRAIN only ----
if CONFIG["drop_high_missing"]:
    thr = CONFIG["high_missing_threshold"]
    protect = {"isFraud", "TransactionID", "TransactionDT", "TransactionAmt",
               "card1", "addr1", "P_emaildomain", "ProductCD"}
    miss_train = train_df.isna().mean()
    drop_cols = [c for c in miss_train.index if miss_train[c] > thr and c not in protect]
    print(f"Dropping {len(drop_cols)} high-missing cols (missingness computed on TRAIN only, thr={thr})")
    for d in (train_df, val_df, test_df):
        d.drop(columns=[c for c in drop_cols if c in d.columns], inplace=True)


In [ ]:
def build_feature_list(frame, include_behavioral=True, include_fraud_history=False):
    core = [c for c in [
        "TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain",
        "DeviceType", "DeviceInfo",
    ] if c in frame.columns]
    groups = []
    for prefix in ("C", "D", "V", "M", "id_"):
        groups += [c for c in frame.columns if c.startswith(prefix)]
    beh = [c for c in frame.columns if c.startswith("beh_")] if include_behavioral else []
    if include_behavioral and not include_fraud_history:
        beh = [c for c in beh if c not in ("beh_prev_fraud_rate", "beh_prev_fraud_count")]
    ban = {"TransactionID", "isFraud", "TransactionDT", "entity_key"}
    feats, seen = [], set()
    for c in core + groups + beh:
        if c not in ban and c not in seen and c in frame.columns:
            seen.add(c)
            feats.append(c)
    return feats

FEATURES_EXP0_BASELINE = build_feature_list(train_df, include_behavioral=False)
FEATURES_EXP1_BEHAVIORAL = build_feature_list(train_df, include_behavioral=True, include_fraud_history=False)
FEATURES_EXP1B_LABEL_AWARE = build_feature_list(train_df, include_behavioral=True, include_fraud_history=True)

print("n_features EXP-0 baseline      :", len(FEATURES_EXP0_BASELINE))
print("n_features EXP-1 behavioral    :", len(FEATURES_EXP1_BEHAVIORAL))
print("n_features EXP-1b label-aware  :", len(FEATURES_EXP1B_LABEL_AWARE))

class Preprocessor:
    '''Fit ONLY on TRAIN. Categorical -> frequency encoding, numeric -> median impute,
    high-missingness (within cols, secondary check) -> missing-flag column. All statistics
    (frequencies, medians, missing-rate) come exclusively from the data passed to .fit().'''

    def __init__(self, cols, miss_thr=0.5):
        self.cols = cols
        self.miss_thr = miss_thr
        self.cat_cols, self.num_cols, self.flag_cols = [], [], []
        self.freq, self.med = {}, {}

    def fit(self, d):
        self.cols = [c for c in self.cols if c in d.columns]
        self.cat_cols = [c for c in self.cols if d[c].dtype == object or str(d[c].dtype) == "category"]
        self.num_cols = [c for c in self.cols if c not in self.cat_cols]
        miss = d[self.cols].isna().mean()
        self.flag_cols = miss[miss > self.miss_thr].index.tolist()
        for c in self.cat_cols:
            self.freq[c] = d[c].astype(str).value_counts(normalize=True).to_dict()
        for c in self.num_cols:
            self.med[c] = float(d[c].median()) if d[c].notna().any() else 0.0
        return self

    def transform(self, d):
        n = len(d)
        data = {}
        for c in self.flag_cols:
            data[f"{c}_miss"] = d[c].isna().astype(np.float32).values if c in d.columns else np.zeros(n, np.float32)
        for c in self.cat_cols:
            m = self.freq.get(c, {})
            data[c] = d[c].astype(str).map(m).fillna(0).astype(np.float32).values if c in d.columns else np.zeros(n, np.float32)
        for c in self.num_cols:
            med = self.med.get(c, 0.0)
            data[c] = d[c].fillna(med).astype(np.float32).values if c in d.columns else np.full(n, med, np.float32)
        return pd.DataFrame(data, index=d.index)

y_train = train_df["isFraud"].values.astype(np.int8)
y_val = val_df["isFraud"].values.astype(np.int8)
y_test = test_df["isFraud"].values.astype(np.int8)

# EXP-0 baseline matrices
prep_exp0 = Preprocessor(FEATURES_EXP0_BASELINE).fit(train_df)
X_train_exp0 = prep_exp0.transform(train_df)
X_val_exp0 = prep_exp0.transform(val_df)
X_test_exp0 = prep_exp0.transform(test_df)

# EXP-1 behavioral matrices (MAIN pipeline used for all downstream sections)
prep_exp1 = Preprocessor(FEATURES_EXP1_BEHAVIORAL).fit(train_df)
X_train = prep_exp1.transform(train_df)
X_val = prep_exp1.transform(val_df)
X_test = prep_exp1.transform(test_df)

print("EXP-0", X_train_exp0.shape, X_val_exp0.shape, X_test_exp0.shape)
print("EXP-1", X_train.shape, X_val.shape, X_test.shape, f"RSS={rss_mb():.0f}")


## 5. Metrics helpers (VAL-tuned threshold)

**Do not use thr=0.5 for F1/MCC on this imbalance** — probabilities are often all < 0.5 → fake zeros.

In [ ]:
def best_f1_threshold(y_true, proba):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    if len(thr) == 0:
        return 0.5
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    return float(thr[int(np.nanargmax(f1))])

def metrics_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "PR_AUC": float(average_precision_score(y_true, proba)),
        "ROC_AUC": float(roc_auc_score(y_true, proba)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "MCC": float(matthews_corrcoef(y_true, pred)),
        "threshold": float(thr),
    }

def fc_at_budget(y_true, proba, fracs=(0.01, 0.05, 0.10)):
    order = np.argsort(-proba)
    y_s = y_true[order]
    tot = max(float(y_true.sum()), 1.0)
    out = {}
    for b in fracs:
        k = max(int(np.ceil(len(y_true) * b)), 1)
        out[f"FC@{int(b*100)}%"] = float(y_s[:k].sum() / tot)
    return out

pos_w = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print("pos_weight", round(pos_w, 2))


## 6. Multi-model × 5-seed runs → mean ± std (EXP-1 main pipeline)

In [ ]:
def make_models(seed, pos_weight):
    models = {}
    if HAS_LGB:
        models["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=600, num_leaves=63, learning_rate=0.03,
            subsample=0.85, colsample_bytree=0.7, min_child_samples=30,
            reg_lambda=1.5, scale_pos_weight=pos_weight,
            random_state=seed, n_jobs=-1, verbose=-1,
        )
    if HAS_XGB:
        models["XGBoost"] = xgb.XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.03,
            subsample=0.85, colsample_bytree=0.7, min_child_weight=5,
            reg_lambda=2.0, scale_pos_weight=pos_weight,
            eval_metric="aucpr", random_state=seed, n_jobs=-1, tree_method="hist",
        )
    models["RandomForest"] = RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=20,
        class_weight="balanced_subsample", max_features="sqrt",
        random_state=seed, n_jobs=-1,
    )
    return models

def run_multimodel(X_tr, y_tr, X_va, y_va, X_te, y_te, seeds, tag, pos_weight):
    '''Fits models[seed], stores VAL *and* TEST probabilities (Issue 7), and reports
    TEST metrics at the VAL-tuned F1 threshold.'''
    rows = []
    proba_val_store, proba_test_store = {}, {}
    for seed in seeds:
        print(f"\n===== [{tag}] SEED {seed} =====", flush=True)
        for name, model in make_models(seed, pos_weight).items():
            print(f"  fit {name}...", flush=True)
            t0 = time.time()
            if name == "LightGBM":
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    eval_metric="average_precision",
                    callbacks=[lgb.early_stopping(50, verbose=False)],
                )
            else:
                model.fit(X_tr, y_tr)
            pv = model.predict_proba(X_va)[:, 1]
            pt = model.predict_proba(X_te)[:, 1]
            thr = best_f1_threshold(y_va, pv)
            m = metrics_at_threshold(y_te, pt, thr)
            m.update(fc_at_budget(y_te, pt))
            m["val_PR_AUC"] = float(average_precision_score(y_va, pv))
            m["seed"] = seed
            m["model"] = name
            m["tag"] = tag
            m["fit_s"] = round(time.time() - t0, 1)
            rows.append(m)
            proba_val_store[(name, seed)] = pv
            proba_test_store[(name, seed)] = pt
            print(f"    thr={thr:.3f} PR-AUC={m['PR_AUC']:.4f} ROC={m['ROC_AUC']:.4f} "
                  f"F1={m['F1']:.4f} MCC={m['MCC']:.4f} Rec={m['Recall']:.4f}")
            gc.collect()
    return pd.DataFrame(rows), proba_val_store, proba_test_store

def summarize_mean_std(df_raw, metric_cols, tag):
    wide = []
    print("=" * 72)
    print(f"[{tag}] TEST mean ± std by model (5 seeds, VAL max-F1 threshold)")
    print("=" * 72)
    for model_name, g in df_raw.groupby("model"):
        row = {"model": model_name, "tag": tag}
        print(f"\n### {model_name}")
        for c in metric_cols:
            mu, sd = g[c].mean(), g[c].std(ddof=1)
            s = f"{mu:.4f} ± {sd:.4f}"
            row[c] = s
            row[f"{c}__mean"] = mu
            print(f"  {c:12s}  {s}")
        wide.append(row)
    return pd.DataFrame(wide)

METRIC_COLS = ["PR_AUC", "ROC_AUC", "Precision", "Recall", "F1", "MCC",
               "FC@1%", "FC@5%", "FC@10%", "threshold"]

# ---- EXP-1 (main): behavioral features, deployment-realistic (no fraud-label history) ----
df_raw_exp1, val_store_exp1, test_store_exp1 = run_multimodel(
    X_train, y_train, X_val, y_val, X_test, y_test, SEEDS, "EXP1_behavioral", pos_w
)
df_raw_exp1.to_csv(f"{CONFIG['output_dir']}/EXP1_behavioral_raw.csv", index=False)
df_mean_std_exp1 = summarize_mean_std(df_raw_exp1, METRIC_COLS, "EXP1_behavioral")
df_mean_std_exp1.to_csv(f"{CONFIG['output_dir']}/EXP1_behavioral_mean_std.csv", index=False)
print("\n")
display(df_mean_std_exp1[["model", "tag"] + METRIC_COLS])


## 6b. EXP-0 — baseline ablation (native IEEE-CIS features, no behavioral features)

Same models, same seeds, same VAL-tuned threshold protocol, applied to `FEATURES_EXP0_BASELINE`.
This answers: *does the leakage-safe behavioral/velocity feature block actually help?* (Issue 3)

In [ ]:
if CONFIG["RUN_BASELINE_ABLATION"]:
    df_raw_exp0, val_store_exp0, test_store_exp0 = run_multimodel(
        X_train_exp0, y_train, X_val_exp0, y_val, X_test_exp0, y_test, SEEDS, "EXP0_baseline", pos_w
    )
    df_raw_exp0.to_csv(f"{CONFIG['output_dir']}/EXP0_baseline_raw.csv", index=False)
    df_mean_std_exp0 = summarize_mean_std(df_raw_exp0, METRIC_COLS, "EXP0_baseline")
    df_mean_std_exp0.to_csv(f"{CONFIG['output_dir']}/EXP0_baseline_mean_std.csv", index=False)
    print("\n")
    display(df_mean_std_exp0[["model", "tag"] + METRIC_COLS])
else:
    df_raw_exp0 = df_mean_std_exp0 = None
    print("Baseline ablation skipped (CONFIG['RUN_BASELINE_ABLATION']=False)")


In [ ]:
# ---- Δ table: EXP-1 (behavioral) − EXP-0 (baseline), mean over 5 seeds, per model ----
if CONFIG["RUN_BASELINE_ABLATION"]:
    delta_metric_cols = ["PR_AUC", "ROC_AUC", "F1", "MCC", "Recall", "FC@1%", "FC@5%", "FC@10%"]
    delta_rows = []
    for model_name in sorted(set(df_raw_exp1["model"]) & set(df_raw_exp0["model"])):
        row = {"model": model_name}
        g1 = df_raw_exp1[df_raw_exp1["model"] == model_name]
        g0 = df_raw_exp0[df_raw_exp0["model"] == model_name]
        for c in delta_metric_cols:
            row[f"delta_{c}"] = round(float(g1[c].mean() - g0[c].mean()), 5)
        delta_rows.append(row)
    df_delta = pd.DataFrame(delta_rows)
    df_delta.to_csv(f"{CONFIG['output_dir']}/EXP1_minus_EXP0_delta.csv", index=False)
    print("Δ (behavioral − baseline), mean of 5 seeds, TEST:")
    display(df_delta)
    print('\nInterpretation: a small positive delta in PR-AUC / FC@budget supports the claim that '
          'leakage-safe behavioral/velocity context provides incremental ranking value on top of native '
          'transaction features -- the interesting result is expected to be modest, not revolutionary; '
          'the larger operational gain in this study comes from the capacity- and cost-aware action '
          'policy in Section 9, not from feature engineering alone.\n')


## 6c. Optional — label-aware behavioral sensitivity experiment

Uses `FEATURES_EXP1B_LABEL_AWARE` (adds `beh_prev_fraud_rate` / `beh_prev_fraud_count`).
This is **not** the main result — it exists only to quantify how much of the behavioral gain would be
attributable to an unrealistic "instant label" assumption (Issue 4). Runs a single seed / LightGBM only
to keep runtime bounded; widen `SEEDS`/models if you want full mean±std here too.

In [ ]:
if CONFIG["RUN_FRAUD_LABEL_SENSITIVITY"] and HAS_LGB:
    prep_exp1b = Preprocessor(FEATURES_EXP1B_LABEL_AWARE).fit(train_df)
    X_train_1b = prep_exp1b.transform(train_df)
    X_val_1b = prep_exp1b.transform(val_df)
    X_test_1b = prep_exp1b.transform(test_df)

    m = lgb.LGBMClassifier(
        n_estimators=600, num_leaves=63, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.7, min_child_samples=30,
        reg_lambda=1.5, scale_pos_weight=pos_w, random_state=42, n_jobs=-1, verbose=-1,
    )
    m.fit(X_train_1b, y_train, eval_set=[(X_val_1b, y_val)],
          eval_metric="average_precision", callbacks=[lgb.early_stopping(50, verbose=False)])
    pv_1b = m.predict_proba(X_val_1b)[:, 1]
    pt_1b = m.predict_proba(X_test_1b)[:, 1]
    thr_1b = best_f1_threshold(y_val, pv_1b)
    m_1b = metrics_at_threshold(y_test, pt_1b, thr_1b)
    m_1b.update(fc_at_budget(y_test, pt_1b))

    lgbm_row = df_raw_exp1[(df_raw_exp1["model"] == "LightGBM") & (df_raw_exp1["seed"] == 42)].iloc[0]
    print("LightGBM (seed=42), deployment-realistic (no fraud-label history):")
    print(f"  PR-AUC={lgbm_row['PR_AUC']:.4f}  FC@5%={lgbm_row['FC@5%']:.4f}")
    print("LightGBM (seed=42), label-aware sensitivity (WITH beh_prev_fraud_rate/count):")
    print(f"  PR-AUC={m_1b['PR_AUC']:.4f}  FC@5%={m_1b['FC@5%']:.4f}")
    print(f"  Δ PR-AUC (label-aware − realistic) = {m_1b['PR_AUC'] - lgbm_row['PR_AUC']:+.4f}")
    print("\nIf this delta is large, it is evidence the main pipeline is right to exclude the feature: a large\n"
          "chunk of 'behavioral' lift would otherwise be coming from an assumption that isn't realistic in most\n"
          "deployment settings (fraud labels confirmed within milliseconds of the previous transaction).")
else:
    print("Fraud-label sensitivity experiment skipped.")


## 7. Soft ensemble — calibrated on VAL, evaluated once on TEST (EXP-1 main pipeline)

In [ ]:
# FIX (Issue 7): use stored VAL probabilities to build the ensemble's VAL score, tune the
# threshold there, freeze it, and only then apply it to the ensemble's TEST score.
model_names = sorted({k[0] for k in val_store_exp1})

model_avg_val = {name: np.mean(np.vstack([val_store_exp1[(name, s)] for s in SEEDS if (name, s) in val_store_exp1]), axis=0)
                  for name in model_names}
model_avg_test = {name: np.mean(np.vstack([test_store_exp1[(name, s)] for s in SEEDS if (name, s) in test_store_exp1]), axis=0)
                   for name in model_names}

ens_val_proba = np.mean(np.vstack([model_avg_val[n] for n in model_names]), axis=0)
ens_test_proba = np.mean(np.vstack([model_avg_test[n] for n in model_names]), axis=0)

ens_thr = best_f1_threshold(y_val, ens_val_proba)   # frozen on VAL
ens = metrics_at_threshold(y_test, ens_test_proba, ens_thr)
ens.update(fc_at_budget(y_test, ens_test_proba))
print("ENSEMBLE (seed-avg then model-avg), threshold tuned on VAL, applied to TEST:")
for k, v in ens.items():
    print(f"  {k}: {v if not isinstance(v, float) else round(v, 4)}")

# Best single model selected by mean VAL PR-AUC (NOT test) to avoid any test-set snooping
# in model selection.
val_pr_auc_by_model = (
    df_raw_exp1.groupby("model")["val_PR_AUC"].mean().sort_values(ascending=False)
)
best_model = val_pr_auc_by_model.index[0]
print("\nBest model by mean VAL PR-AUC:", best_model, f"({val_pr_auc_by_model.iloc[0]:.4f})")
proba_best_val = model_avg_val[best_model]
proba_best_test = model_avg_test[best_model]


## 8. PCA + UMAP — illustrative geometry (not performance proof)

In [ ]:
rng = np.random.RandomState(42)
fraud_idx = np.where(y_test == 1)[0]
legit_idx = np.where(y_test == 0)[0]
n_fraud_viz = min(len(fraud_idx), 2000)
n_legit_viz = min(len(legit_idx), 6000)
viz_idx = np.concatenate([
    rng.choice(fraud_idx, n_fraud_viz, replace=False),
    rng.choice(legit_idx, n_legit_viz, replace=False),
])
rng.shuffle(viz_idx)

X_viz = X_test.iloc[viz_idx]
y_viz = y_test[viz_idx]
risk_viz = ens_test_proba[viz_idx]

scaler_v = StandardScaler()
X_scaled = scaler_v.fit_transform(X_viz)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA var explained (2D): {pca.explained_variance_ratio_.sum()*100:.1f}%")

leg = y_viz == 0
fr = y_viz == 1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.scatter(X_pca[leg, 0], X_pca[leg, 1], c="#4C72B0", s=8, alpha=0.25, label="Legit", rasterized=True)
ax.scatter(X_pca[fr, 0], X_pca[fr, 1], c="#C44E52", s=16, alpha=0.75, label="Fraud", rasterized=True)
ax.set_title("PCA — fraud vs legit")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.legend(markerscale=2)

ax = axes[1]
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=risk_viz, cmap="magma", s=10, alpha=0.6, rasterized=True)
plt.colorbar(sc, ax=ax, fraction=0.046, label="P(fraud)")
ax.set_title("PCA — colored by ensemble risk")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/pca_fraud.png", dpi=150, bbox_inches="tight")
plt.show()
print("Reminder: this is an illustrative low-dimensional projection, not evidence of a separable cluster.")


In [ ]:
X_umap = None
if HAS_UMAP:
    print("UMAP fitting...", flush=True)
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.15, n_components=2,
                        metric="euclidean", random_state=42, low_memory=True)
    X_umap = reducer.fit_transform(X_scaled)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ax = axes[0]
    ax.scatter(X_umap[leg, 0], X_umap[leg, 1], c="#4C72B0", s=8, alpha=0.25, label="Legit", rasterized=True)
    ax.scatter(X_umap[fr, 0], X_umap[fr, 1], c="#C44E52", s=18, alpha=0.8, label="Fraud", rasterized=True)
    ax.set_title("UMAP — fraud vs legit")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.legend(markerscale=2)

    ax = axes[1]
    sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=risk_viz, cmap="magma", s=10, alpha=0.65, rasterized=True)
    plt.colorbar(sc, ax=ax, fraction=0.046, label="P(fraud)")
    ax.set_title("UMAP — colored by ensemble risk")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/umap_fraud.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("!pip install umap-learn  then re-run this cell")


In [ ]:
def fraud_ellipse(ax, XY, y, label="Fraud region"):
    pts = XY[y == 1]
    if len(pts) < 10:
        return
    cov = EmpiricalCovariance().fit(pts)
    vals, vecs = np.linalg.eigh(cov.covariance_)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * 2 * np.sqrt(np.maximum(vals, 1e-9))
    center = pts.mean(axis=0)
    ell = Ellipse(xy=center, width=width, height=height, angle=theta,
                  edgecolor="#C44E52", facecolor="#C44E52", alpha=0.15, lw=2, label=label)
    ax.add_patch(ell)
    ax.scatter([center[0]], [center[1]], c="#C44E52", s=40, marker="x", zorder=5)

n_plots = 2 if X_umap is not None else 1
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

ax = axes[0]
ax.scatter(X_pca[leg, 0], X_pca[leg, 1], c="#4C72B0", s=6, alpha=0.2, rasterized=True)
ax.scatter(X_pca[fr, 0], X_pca[fr, 1], c="#C44E52", s=14, alpha=0.7, rasterized=True)
fraud_ellipse(ax, X_pca, y_viz)
ax.set_title("PCA + fraud covariance ellipse")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(loc="best")

if X_umap is not None:
    ax = axes[1]
    ax.scatter(X_umap[leg, 0], X_umap[leg, 1], c="#4C72B0", s=6, alpha=0.2, rasterized=True)
    ax.scatter(X_umap[fr, 0], X_umap[fr, 1], c="#C44E52", s=14, alpha=0.7, rasterized=True)
    fraud_ellipse(ax, X_umap, y_viz)
    ax.set_title("UMAP + fraud covariance ellipse")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.legend(loc="best")

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/fraud_ellipse.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fraud_ellipse.png")


## 9. Capacity- and cost-aware APPROVE / REVIEW / BLOCK policy

**Fix for Issue 1 (test-set policy optimization) and Issue 2 (no review-capacity constraint):**

- `optimize_thresholds_capacity()` searches `(tau1, tau2)` on **VAL** subject to `review_rate ≤ budget`.
- The resulting thresholds are **frozen** and evaluated exactly once on **TEST**.
- We sweep a 3×3 grid: 3 cost scenarios (`FN cost = 10×/25×/50× FP cost`) × 3 review-capacity budgets
  (1% / 5% / 10%), plus a fixed `p > 0.5` baseline for comparison.

In [ ]:
def expected_cost(y_true, proba, tau1, tau2, c_fn, c_fp, c_review):
    action = np.where(proba < tau1, "APPROVE", np.where(proba < tau2, "REVIEW", "BLOCK"))
    cost = np.zeros(len(y_true))
    cost[(action == "APPROVE") & (y_true == 1)] = c_fn
    cost[(action == "BLOCK") & (y_true == 0)] = c_fp
    cost[action == "REVIEW"] = c_review
    return {
        "tau1": float(tau1), "tau2": float(tau2), "total_cost": float(cost.sum()),
        "fraud_capture": float(y_true[action != "APPROVE"].sum() / max(y_true.sum(), 1)),
        "review_rate": float((action == "REVIEW").mean()),
        "false_decline_rate": float(((action == "BLOCK") & (y_true == 0)).sum() / max((y_true == 0).sum(), 1)),
    }

def optimize_thresholds_capacity(y_true, proba, c_fn, c_fp, c_review, max_review_rate=None, grid=41):
    '''Minimize expected cost over (tau1, tau2) subject to review_rate <= max_review_rate.
    Falls back to the unconstrained optimum (flagged infeasible_capacity=True) only if no
    grid point satisfies the constraint, which should not normally happen for budgets >= 1%.'''
    taus = np.linspace(0.01, 0.99, grid)
    best = None
    for t1 in taus:
        for t2 in taus:
            if t2 <= t1:
                continue
            r = expected_cost(y_true, proba, t1, t2, c_fn, c_fp, c_review)
            if max_review_rate is not None and r["review_rate"] > max_review_rate + 1e-9:
                continue
            if best is None or r["total_cost"] < best["total_cost"]:
                best = r
    if best is None:
        best = optimize_thresholds_capacity(y_true, proba, c_fn, c_fp, c_review, None, grid)
        best["infeasible_capacity"] = True
    else:
        best["infeasible_capacity"] = False
    return best

C_FP, C_REVIEW = CONFIG["C_FP"], CONFIG["C_REVIEW"]
policy_rows = []
print("Policy thresholds optimized on VAL, frozen, evaluated once on TEST.\n")
for scen_name, c_fn in CONFIG["COST_SCENARIOS"].items():
    # fixed p>0.5 baseline (no optimization) — evaluated directly on TEST for comparison
    conv = expected_cost(y_test, ens_test_proba, 0.5, 0.5 + 1e-9, c_fn, C_FP, C_REVIEW)
    conv.update({"scenario": scen_name, "capacity_budget": "n/a", "policy": "p_gt_0.5"})
    policy_rows.append(conv)

    # unconstrained cost-optimum, for reference (still VAL -> TEST, no capacity limit)
    opt_unc_val = optimize_thresholds_capacity(y_val, ens_val_proba, c_fn, C_FP, C_REVIEW, None)
    opt_unc_test = expected_cost(y_test, ens_test_proba, opt_unc_val["tau1"], opt_unc_val["tau2"], c_fn, C_FP, C_REVIEW)
    opt_unc_test.update({"scenario": scen_name, "capacity_budget": "unconstrained", "policy": "cost_optimized_VAL_to_TEST"})
    policy_rows.append(opt_unc_test)

    for budget in CONFIG["REVIEW_CAPACITY_BUDGETS"]:
        opt_val = optimize_thresholds_capacity(y_val, ens_val_proba, c_fn, C_FP, C_REVIEW, max_review_rate=budget)
        opt_test = expected_cost(y_test, ens_test_proba, opt_val["tau1"], opt_val["tau2"], c_fn, C_FP, C_REVIEW)
        opt_test.update({
            "scenario": scen_name,
            "capacity_budget": f"{int(budget*100)}%",
            "policy": "cost_capacity_optimized_VAL_to_TEST",
            "infeasible_capacity": opt_val["infeasible_capacity"],
        })
        policy_rows.append(opt_test)
        print(f"{scen_name} | capacity<={int(budget*100):>2d}% : "
              f"cost={opt_test['total_cost']:.0f}  capture={opt_test['fraud_capture']:.3f}  "
              f"review_rate={opt_test['review_rate']:.3f}  (tau1={opt_test['tau1']:.3f}, tau2={opt_test['tau2']:.3f})")
    print()

df_policy = pd.DataFrame(policy_rows)
df_policy.to_csv(f"{CONFIG['output_dir']}/policy_grid.csv", index=False)
display(df_policy[["scenario", "capacity_budget", "policy", "tau1", "tau2",
                    "total_cost", "fraud_capture", "review_rate", "false_decline_rate"]])


In [ ]:
# ---- chart: capacity-constrained cost vs p>0.5 baseline, faceted by cost scenario ----
plot_df = df_policy[df_policy["capacity_budget"] != "unconstrained"].copy()
budget_order = ["1%", "5%", "10%", "n/a"]
plot_df["capacity_budget"] = pd.Categorical(plot_df["capacity_budget"], categories=budget_order, ordered=True)

g = sns.catplot(
    data=plot_df, kind="bar", x="capacity_budget", y="total_cost",
    hue="policy", col="scenario", height=4, aspect=1.0, sharey=False,
)
g.set_titles("{col_name}")
g.set_axis_labels("Review capacity budget", "Total cost (TEST)")
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/policy_cost_by_capacity.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nFC@budget (ensemble, TEST, for reference):")
for k, v in fc_at_budget(y_test, ens_test_proba).items():
    print(f"  {k}: {v:.3f}")


## 10. SHAP — global drivers + representative transactions

Fits one LightGBM on the EXP-1 (behavioral, deployment-realistic) training set. In addition to the global
summary plots, we pull out a **high / medium / low risk** transaction from the TEST sample and report
`P(fraud)`, the resulting action (using the `S2_25x`, 5%-capacity frozen policy from Section 9 as the
illustrative operating point), and the top SHAP drivers for each — this is what makes the explanation
operationally legible rather than a generic global-importance plot.

In [ ]:
if HAS_SHAP and HAS_LGB:
    print("Fitting one LGBM for SHAP...", flush=True)
    shap_model = lgb.LGBMClassifier(
        n_estimators=400, num_leaves=63, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.7, scale_pos_weight=pos_w,
        random_state=42, n_jobs=-1, verbose=-1,
    )
    shap_model.fit(X_train, y_train)
    sample = X_test.sample(n=min(1200, len(X_test)), random_state=42)
    sample_proba = shap_model.predict_proba(sample)[:, 1]
    explainer = shap.TreeExplainer(shap_model)
    sv = explainer.shap_values(sample)
    if isinstance(sv, list):
        sv = sv[1]
    shap.summary_plot(sv, sample, plot_type="bar", show=True)
    shap.summary_plot(sv, sample, show=True)

    # frozen operating point for illustration: S2_25x scenario @ 5% capacity budget
    illus = df_policy[(df_policy["scenario"] == "S2_25x") & (df_policy["capacity_budget"] == "5%")].iloc[0]
    tau1_illus, tau2_illus = illus["tau1"], illus["tau2"]

    def action_for(p):
        if p < tau1_illus:
            return "APPROVE"
        elif p < tau2_illus:
            return "REVIEW"
        return "BLOCK"

    order = np.argsort(-sample_proba)
    n = len(sample_proba)
    picks = {"high_risk": order[0], "medium_risk": order[n // 2], "low_risk": order[-1]}

    rows = []
    for label, idx in picks.items():
        p = float(sample_proba[idx])
        contrib = pd.Series(sv[idx], index=sample.columns)
        top_pos = contrib.sort_values(ascending=False).head(3)
        top_neg = contrib.sort_values(ascending=True).head(3)
        rows.append({
            "case": label,
            "P(fraud)": round(p, 4),
            "action": action_for(p),
            "top_positive_drivers": "; ".join(f"{k}={v:+.3f}" for k, v in top_pos.items()),
            "top_negative_drivers": "; ".join(f"{k}={v:+.3f}" for k, v in top_neg.items()),
        })
    df_shap_cases = pd.DataFrame(rows)
    df_shap_cases.to_csv(f"{CONFIG['output_dir']}/shap_representative_cases.csv", index=False)
    print(f"\nIllustrative operating point: S2_25x scenario, 5% review capacity "
          f"(tau1={tau1_illus:.3f}, tau2={tau2_illus:.3f})\n")
    display(df_shap_cases)
else:
    print("SHAP skipped (need shap + lightgbm)")
    df_shap_cases = None


## 11. Optional — SMOTE sensitivity experiment

The study description mentioned SMOTE but it wasn't present in the pre-final notebook (Issue 8). Added
here as an explicit secondary experiment, applied **only to TRAIN**, LightGBM, single seed (runtime), and
compared against the main class-weighting approach already reported in Section 6.

This is framed as a sensitivity check, not a replacement for `scale_pos_weight` — the expectation
(per the study description) is that SMOTE may raise precision/PR-AUC at some recall cost, not that it
universally wins.

In [ ]:
if CONFIG["RUN_SMOTE_SENSITIVITY"] and HAS_IMBLEARN and HAS_LGB:
    print("Applying SMOTE to TRAIN only...", flush=True)
    sm = SMOTE(random_state=42, n_jobs=-1)
    X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
    print(f"  TRAIN before: {len(y_train):,} (fraud={y_train.mean()*100:.3f}%) "
          f"-> after SMOTE: {len(y_train_sm):,} (fraud={y_train_sm.mean()*100:.3f}%)")

    # no scale_pos_weight here — SMOTE already balances the classes
    m_smote = lgb.LGBMClassifier(
        n_estimators=600, num_leaves=63, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.7, min_child_samples=30,
        reg_lambda=1.5, random_state=42, n_jobs=-1, verbose=-1,
    )
    m_smote.fit(X_train_sm, y_train_sm, eval_set=[(X_val, y_val)],
                eval_metric="average_precision", callbacks=[lgb.early_stopping(50, verbose=False)])
    pv_sm = m_smote.predict_proba(X_val)[:, 1]
    pt_sm = m_smote.predict_proba(X_test)[:, 1]
    thr_sm = best_f1_threshold(y_val, pv_sm)
    m_sm_metrics = metrics_at_threshold(y_test, pt_sm, thr_sm)
    m_sm_metrics.update(fc_at_budget(y_test, pt_sm))

    lgbm_row = df_raw_exp1[(df_raw_exp1["model"] == "LightGBM") & (df_raw_exp1["seed"] == 42)].iloc[0]
    df_smote_compare = pd.DataFrame([
        {"approach": "class_weight (scale_pos_weight)", "PR_AUC": lgbm_row["PR_AUC"],
         "Precision": lgbm_row["Precision"], "Recall": lgbm_row["Recall"], "F1": lgbm_row["F1"]},
        {"approach": "SMOTE (train only)", "PR_AUC": m_sm_metrics["PR_AUC"],
         "Precision": m_sm_metrics["Precision"], "Recall": m_sm_metrics["Recall"], "F1": m_sm_metrics["F1"]},
    ])
    df_smote_compare.to_csv(f"{CONFIG['output_dir']}/smote_sensitivity.csv", index=False)
    print("\nLightGBM, seed=42, TEST — class-weighting vs SMOTE:")
    display(df_smote_compare)
else:
    df_smote_compare = None
    print("SMOTE sensitivity experiment skipped (disabled, or imbalanced-learn/LightGBM unavailable).")


## 12. Final JSON export

In [ ]:
summary = {
    "meta": {
        "sample_n_ieee": CONFIG["SAMPLE_N_IEEE"],
        "sample_n_ieee_note": (
            "Head-of-file sample; because the split is temporal this under-represents later "
            "transactions. Set SAMPLE_N_IEEE=None for the final full-data run if feasible."
        ),
        "seeds": SEEDS,
        "n_features_exp0_baseline": len(FEATURES_EXP0_BASELINE),
        "n_features_exp1_behavioral": len(FEATURES_EXP1_BEHAVIORAL),
        "threshold_rule": "VAL max-F1, applied to TEST",
        "cost_policy_rule": "tau1/tau2 optimized on VAL under a review-capacity constraint, frozen, evaluated once on TEST",
        "historical_fraud_labels_in_main_pipeline": CONFIG["USE_HISTORICAL_FRAUD_LABELS_MAIN"],
        "paysim": "Not present in this notebook; would need to be reported as a separate cross-dataset transfer study, not claimed here.",
        "smote": "Optional sensitivity experiment (Section 11); main pipeline uses class weighting.",
    },
    "exp1_behavioral_mean_std_by_model": df_mean_std_exp1.drop(columns=[c for c in df_mean_std_exp1.columns if c.endswith("__mean")]).to_dict(orient="records"),
    "exp0_baseline_mean_std_by_model": (
        df_mean_std_exp0.drop(columns=[c for c in df_mean_std_exp0.columns if c.endswith("__mean")]).to_dict(orient="records")
        if CONFIG["RUN_BASELINE_ABLATION"] else None
    ),
    "behavioral_ablation_delta": df_delta.to_dict(orient="records") if CONFIG["RUN_BASELINE_ABLATION"] else None,
    "ensemble_test_val_calibrated_threshold": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in ens.items()},
    "best_model_by_mean_val_pr_auc": best_model,
    "capacity_cost_policy_grid": df_policy.to_dict(orient="records"),
    "shap_representative_cases": df_shap_cases.to_dict(orient="records") if df_shap_cases is not None else None,
    "smote_sensitivity": df_smote_compare.to_dict(orient="records") if df_smote_compare is not None else None,
}
path = Path(CONFIG["output_dir"]) / "final_summary.json"
with open(path, "w") as f:
    json.dump(summary, f, indent=2)
print("Saved", path)
print("Artifacts in", CONFIG["output_dir"])
print(os.listdir(CONFIG["output_dir"]))


## How to run

1. Optional: `!pip install umap-learn shap imbalanced-learn`
2. Run all cells top → bottom
3. For the final paper run on full data: set `CONFIG["SAMPLE_N_IEEE"] = None` (high-RAM env; RF may be slow — consider dropping RF or subsampling only RF's training rows for that run)
4. Toggle `RUN_BASELINE_ABLATION`, `RUN_FRAUD_LABEL_SENSITIVITY`, `RUN_SMOTE_SENSITIVITY` in `CONFIG` to control runtime vs completeness

**Interpretation reminder**
- ROC-AUC ~0.93 = strong ranking
- PR-AUC ~0.55 = normal under temporal split + rare fraud
- MCC/F1 only valid with VAL-tuned threshold (this notebook does that)
- PCA/UMAP are illustrative geometry, not performance proof
- **Cost-policy and ensemble thresholds are tuned on VAL and frozen before touching TEST — TEST is touched exactly once per reported number.**
- The behavioral ablation (Section 6b/6c) is what supports the claim "behavioral context helps"; without it, that claim would be unsupported by this notebook alone.
- PaySim and full-data runs are **not** included here; if used elsewhere, report them explicitly as separate cross-dataset/scale sensitivity analyses, not as part of this notebook's primary result.
